In [1]:
import os
import pandas as pd

# Path to our raw data folder (going one level up from 'notebooks' into 'data/raw')
RAW_DATA_DIR = os.path.join("..", "data", "raw")

# Check what files are in that folder
files = os.listdir(RAW_DATA_DIR)
print("Files found in data/raw:")
for f in files:
    print(f" - {f}")

Files found in data/raw:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_orders_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


In [2]:
# Create a neat summary of all our datasets
summary_list = []

for filename in files:
    if filename.endswith(".csv"):
        filepath = os.path.join(RAW_DATA_DIR, filename)
        df = pd.read_csv(filepath)
        
        summary_list.append({
            "Dataset Name": filename,
            "Total Rows": f"{df.shape[0]:,}",
            "Total Columns": df.shape[1],
            "Missing Values": f"{df.isnull().sum().sum():,}"
        })

# Display as a clean, styled table right inside Jupyter!
summary_table = pd.DataFrame(summary_list)
summary_table

,Dataset Name,Total Rows,Total Columns,Missing Values
0,olist_customers_dataset.csv,"99,441",5,0
1,olist_geolocation_dataset.csv,"1,000,163",5,0
2,olist_orders_dataset.csv,"99,441",8,"4,908"
3,olist_order_items_dataset.csv,"112,650",7,0
4,olist_order_payments_dataset.csv,"103,886",5,0
5,olist_order_reviews_dataset.csv,"99,224",7,"145,903"
6,olist_products_dataset.csv,"32,951",9,"2,448"
7,olist_sellers_dataset.csv,"3,095",4,0
8,product_category_name_translation.csv,71,2,0


In [3]:
# Load the 3 core tables
df_customers = pd.read_csv(os.path.join(RAW_DATA_DIR, "olist_customers_dataset.csv"))
df_orders = pd.read_csv(os.path.join(RAW_DATA_DIR, "olist_orders_dataset.csv"))
df_items = pd.read_csv(os.path.join(RAW_DATA_DIR, "olist_order_items_dataset.csv"))

print("--- CUSTOMER TABLE PREVIEW ---")
display(df_customers.head(3))

print("Total customer_id values:", len(df_customers))
print("Total UNIQUE human customers (customer_unique_id):", df_customers["customer_unique_id"].nunique())

--- CUSTOMER TABLE PREVIEW ---


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


Total customer_id values: 99441
Total UNIQUE human customers (customer_unique_id): 96096


In [4]:
# Check order statuses
order_status_counts = df_orders["order_status"].value_counts()
print("--- ORDER STATUS BREAKDOWN ---")
display(order_status_counts)

--- ORDER STATUS BREAKDOWN ---


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [5]:
# Convert purchase date column from text to datetime format
df_orders["order_purchase_timestamp"] = pd.to_datetime(df_orders["order_purchase_timestamp"])

min_date = df_orders["order_purchase_timestamp"].min()
max_date = df_orders["order_purchase_timestamp"].max()

print(f"Earliest Order Date: {min_date}")
print(f"Latest Order Date:   {max_date}")
print(f"Total Timeframe:     {(max_date - min_date).days} days (~2 years)")

Earliest Order Date: 2016-09-04 21:15:19
Latest Order Date:   2018-10-17 17:30:18
Total Timeframe:     772 days (~2 years)


In [6]:
# Load payments table
df_payments = pd.read_csv(os.path.join(RAW_DATA_DIR, "olist_order_payments_dataset.csv"))

print("--- PAYMENT METHODS SUMMARY ---")
payment_summary = df_payments.groupby("payment_type").agg(
    total_transactions=("order_id", "count"),
    total_value=("payment_value", "sum"),
    avg_value=("payment_value", "mean")
).reset_index()

# Sort by highest transaction volume
payment_summary = payment_summary.sort_values(by="total_transactions", ascending=False)
display(payment_summary)

--- PAYMENT METHODS SUMMARY ---


,payment_type,total_transactions,total_value,avg_value
1,credit_card,76795,12542084.19,163.319021
0,boleto,19784,2869361.27,145.034435
4,voucher,5775,379436.87,65.703354
2,debit_card,1529,217989.79,142.570170
3,not_defined,3,0.00,0.000000


In [7]:
# Load products and translations
df_products = pd.read_csv(os.path.join(RAW_DATA_DIR, "olist_products_dataset.csv"))
df_trans = pd.read_csv(os.path.join(RAW_DATA_DIR, "product_category_name_translation.csv"))

# Merge products with English translation
df_products_eng = pd.merge(df_products, df_trans, on="product_category_name", how="left")

# Check top 10 categories
top_categories = df_products_eng["product_category_name_english"].value_counts().head(10)
print("--- TOP 10 PRODUCT CATEGORIES ---")
display(top_categories)

--- TOP 10 PRODUCT CATEGORIES ---


product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

In [8]:
# Load reviews table
df_reviews = pd.read_csv(os.path.join(RAW_DATA_DIR, "olist_order_reviews_dataset.csv"))

print("--- REVIEW SCORE DISTRIBUTION ---")
review_dist = df_reviews["review_score"].value_counts(normalize=True).sort_index(ascending=False) * 100
for score, pct in review_dist.items():
    print(f"Rating {score} Stars: {pct:.1f}%")

print(f"\nAverage Review Score across platform: {df_reviews['review_score'].mean():.2f} / 5.0")

--- REVIEW SCORE DISTRIBUTION ---
Rating 5 Stars: 57.8%
Rating 4 Stars: 19.3%
Rating 3 Stars: 8.2%
Rating 2 Stars: 3.2%
Rating 1 Stars: 11.5%

Average Review Score across platform: 4.09 / 5.0
